# Vigil — Time-Series Forecasting Benchmark
**Chronos-T5-Small vs TimesFM-2.5-200M on Synthetic Network Telemetry**

This notebook evaluates two zero-shot time-series foundation models on three network metric types that Vigil monitors in production: CPU utilisation, packet drop rate, and BGP route count. The goal is to identify which model provides more accurate and better-calibrated 24-step forecasts — a prerequisite for anomaly-aware pre-triage in the Vigil FSM.

| Model | Source | Parameters | Architecture |
|-------|--------|-----------|--------------|
| Chronos-T5-Small | Amazon | ~46M | T5 encoder-decoder, trained on 27B tokens |
| TimesFM-2.5-200M | Google | ~200M | Patched-decoder transformer |

Metrics used: **MASE** (point accuracy vs naïve baseline), **sMAPE** (percentage error), **CRPS** (probabilistic calibration).

<a href="https://colab.research.google.com/github/bnamatherdhala7/Vigil/blob/main/splunk_evals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1 · Environment Check
Prints the Python version to confirm the runtime is 3.10+ (both Chronos and TimesFM require ≥ 3.9).

## 2 · Package Installation — First Pass
Installs three core libraries: `chronos-forecasting` (Amazon time-series foundation model), `timesfm` (Google time-series foundation model), and `properscoring` (probabilistic accuracy metrics). Dependency conflicts with the Colab base image (CUDA 12 vs 13, torch version) are expected and non-fatal — the packages still load correctly.

## 3 · Import Verification
Confirms all three packages import cleanly after installation. If any of these fail, re-run cell 2 before continuing.

## 4 · Synthetic Dataset — Draft
Early prototype of the data generator. Produces 60 time series (20 devices × 3 metric types) representing realistic network telemetry signals:

| Type | Signal shape | Real-world analogue |
|------|-------------|---------------------|
| `cpu` | Sinusoidal daily cycle + random spikes | Router/switch CPU % from Splunk `index=netops` |
| `packet_drop` | Exponential noise baseline + burst windows | Interface error rate via Cisco Catalyst SNMP |
| `bgp` | Step-function flaps between stable route counts | BGP peer state from Splunk MCP `run_spl_query` |

Each series is 536 steps: 512-step context window + 24-step held-out forecast horizon. **This is a draft** — superseded by the cleaner version in cell 12.

## 5 · Evaluation Metric Functions — Draft
Defines the three accuracy metrics used throughout this benchmark. **This is a draft** — superseded by the cleaner version in cell 13.

- **MASE** (Mean Absolute Scaled Error): compares forecast MAE against a naïve seasonal random-walk baseline. MASE < 1.0 means the model beats the baseline.
- **sMAPE** (Symmetric Mean Absolute Percentage Error): percentage error symmetric around zero; interpretable as a percentage for non-technical stakeholders.
- **CRPS** (Continuous Ranked Probability Score): measures accuracy of the full probabilistic distribution (not just the point forecast); rewards well-calibrated uncertainty estimates, which matters when Vigil decides whether to escalate or suppress an alert.

## 6 · Fix: Force-Reinstall Chronos
The initial install returned a stale cached version of `chronos-forecasting`. `--force-reinstall` fetches the latest build (2.2.2) which introduced the `ChronosPipeline` class used in this notebook. After reinstalling, the module is reloaded in-process to avoid needing a full kernel restart.

## 7 · Confirm Chronos Version Post-Reinstall
Uses `pip show` to verify the installed Chronos version from the shell level (not just what Python has imported). Confirms the force-reinstall in cell 6 took effect.

## 8 · Clean Reinstall — All Three Packages
Reinstalls all three dependencies from scratch to ensure a consistent environment before the main benchmark run. This is the last installation step; all cells below assume packages are ready.

## 9 · Verify Chronos Version in Active Python Session
Imports and prints the Chronos version that is actually loaded in memory — confirms the reinstall from cell 8 is reflected in the running kernel, not just on disk.

## 10 · Reinstall Guard (Idempotent)
A final safety reinstall before the benchmark. Running `pip install -q` when the correct versions are already present is a no-op, so this cell can be re-run freely without side effects.

## 11 · Final Import Block & Device Selection
Imports all required libraries and selects the compute device. The benchmark automatically uses GPU (`cuda`) when available — Chronos and TimesFM both support CUDA and inference is ~10–20× faster on GPU than CPU for 60-series workloads.

## 12 · Synthetic Network Telemetry Dataset (Final)
Builds the full benchmark dataset: 60 time series across three network metric types, fixed seed `42` for reproducibility.

| Type | Generator | Signal characteristics |
|------|-----------|------------------------|
| `cpu` | `make_cpu` | 30–55% base + sinusoidal daily pattern (period 1440 steps) + Gaussian noise + 5 random spikes per series |
| `packet_drop` | `make_packet_drop` | Exponential(0.5) baseline + 3 burst windows of 5–30 steps at +5–20% |
| `bgp` | `make_bgp` | Step function at {30, 60, 90} routes with 2 random flap events per series |

**Split:** `context = all_series[:, :512]` (fed to the model), `ground_truth = all_series[:, 512:]` (24-step horizon held out for scoring).

## 13 · Evaluation Metric Functions (Final)
Production-ready scoring functions. All three metrics are standard in the M4/M5 forecasting competition literature and directly applicable to network telemetry anomaly detection.

- **`mase(forecast, actual, insample)`** — MASE < 1.0 means the model outperforms a naïve carry-forward prediction, the implicit baseline for most SNMP polling tools
- **`smape(forecast, actual)`** — percentage error; interpretable when reporting to network operations or management
- **`crps_score(samples, actual)`** — rewards well-calibrated uncertainty; lower CRPS = fewer over-confident escalations or suppressions in Vigil's FSM decision logic
- **`evaluate(...)`** — aggregates all three metrics per series type, groups by `metric_type`, and returns a summary DataFrame

## 14 · Debug: Chronos Module Path
Prints both the version string and the on-disk path of the loaded `chronos` module. Useful in shared Colab environments where `sys.path` might resolve to a stale cached copy instead of the freshly installed package.

## 15 · Load Chronos Model — First Attempt
Initial exploratory load of `ChronosPipeline` to confirm the model downloads correctly from Hugging Face. The full inference loop (with progress logging and metric evaluation) follows in cell 16. This cell can be skipped if cell 16 is run directly.

## 16 · Chronos-T5-Small Inference
Loads **Amazon Chronos-T5-Small** (a T5 encoder-decoder pre-trained on 27 billion time-series tokens from public datasets) and runs probabilistic 24-step forecasts across all 60 series.

Key parameters:
- `num_samples=100` — draws 100 Monte Carlo samples per series; sample mean → point forecast (MASE/sMAPE), full sample set → CRPS
- `dtype=torch.bfloat16` — half-precision inference, ~2× memory reduction with negligible accuracy loss
- Series are fed one at a time as a `(1, context_len)` tensor; no batching needed at this scale

Results are collected into `chronos_point` (shape `60 × 24`) and `chronos_samples` (shape `60 × 100 × 24`) for evaluation.

## 17 · Inspect TimesFM `ForecastConfig` API
Prints the constructor signature for `timesfm.ForecastConfig` before instantiation. The TimesFM API changed across versions; this guards against silent mis-configuration (e.g., passing a deprecated `context_length` parameter that gets silently ignored).

## 18 · Inspect TimesFM `forecast` Method Signature
Confirms argument names (`horizon`, `inputs`) and return types for the TimesFM inference call. The method returns `(point_forecast, quantile_forecasts)` — the quantile axis is later transposed to shape `(n_quantiles, horizon)` and treated as pseudo-samples for CRPS scoring in cell 19.

## 19 · TimesFM-2.5-200M Inference
Loads **Google TimesFM-2.5-200M** (a 200M-parameter patched-decoder transformer pre-trained on Google-internal time-series data) and runs 24-step forecasts across all 60 series.

Key details:
- `.compile()` applies `torch.compile` for optimised GPU execution (JIT-fused kernels)
- `.model.to(DEVICE)` moves weights to the same device as Chronos for a fair comparison
- `quantiles[0][:PRED_LEN, :].T` extracts the quantile matrix and transposes it to `(n_quantiles, horizon)` shape for CRPS — TimesFM outputs 10 quantiles by default (p10 through p90)
- Results collected into `timesfm_point` and `timesfm_samples` with the same shapes as Chronos

## 20 · Benchmark Results Summary
Side-by-side comparison of Chronos-T5-Small vs TimesFM-2.5-200M across all three network metric types and all three scoring metrics (lower is better for all).

**Key findings:**
- **Chronos wins on all three metrics** (MASE, sMAPE, CRPS) averaged across all 60 series
- `packet_drop` sMAPE is high for both models (65 vs 108) — expected, because near-zero baselines inflate percentage errors; MASE and CRPS are the reliable signals for this metric type
- `bgp` MASE = 0.95 for Chronos (< 1.0) — the model beats naïve prediction for step-function state changes, making Chronos the preferred backbone for BGP flap detection in Vigil's pre-triage classifier
- CRPS advantage for Chronos on BGP (1.15 vs 1.72) means better-calibrated uncertainty — directly translates to fewer false escalations when Vigil's FSM decides whether to suppress, remediate, or escalate an alert

In [ ]:
import sys
print(sys.version)  # confirm it's 3.10.x


3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [ ]:
!pip install -q chronos-forecasting
!pip install -q git+https://github.com/google-research/timesfm.git
!pip install -q properscoring

print('Done.')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Done.


In [ ]:
import sys
print(sys.version)

from chronos import ChronosPipeline
print('Chronos import OK')

import timesfm
print('TimesFM import OK')

import properscoring
print('properscoring import OK')

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Chronos import OK
TimesFM import OK
properscoring import OK


In [ ]:
import numpy as np
import pandas as pd
import torch

np.random.seed(42)

N_DEVICES = 20
CTX_LEN   = 512
PRED_LEN  = 24
TOTAL     = CTX_LEN + PRED_LEN

def make_cpu(n, length):
    out = []
    for _ in range(n):
        t        = np.arange(length)
        base     = np.random.uniform(30, 55)
        daily    = 15 * np.sin(2 * np.pi * t / 1440)
        noise    = np.random.normal(0, 3, length)
        spikes   = np.zeros(length)
        idx      = np.random.choice(length, 5, replace=False)
        spikes[idx] = np.random.uniform(20, 45, 5)
        out.append(np.clip(base + daily + noise + spikes, 0, 100))
    return np.array(out)

def make_packet_drop(n, length):
    out = []
    for _ in range(n):
        base = np.random.exponential(0.5, length)
        for s in np.random.choice(length - 30, 3, replace=False):
            base[s:s+np.random.randint(5,30)] += np.random.uniform(5, 20)
        out.append(np.clip(base + np.abs(np.random.normal(0, 0.2, length)), 0, 100))
    return np.array(out)

def make_bgp(n, length):
    out = []
    for _ in range(n):
        base = np.ones(length) * np.random.choice([30, 60, 90])
        for e in np.random.choice(length, 2, replace=False):
            dur = min(np.random.randint(5,15), length - e)
            base[e:e+dur] = np.random.choice([30, 60, 90, 120])
        out.append(np.clip(base + np.random.normal(0, 1.5, length), 10, 180))
    return np.array(out)

all_series  = np.vstack([make_cpu(N_DEVICES, TOTAL),
                          make_packet_drop(N_DEVICES, TOTAL),
                          make_bgp(N_DEVICES, TOTAL)])

series_meta = ['cpu']*N_DEVICES + ['packet_drop']*N_DEVICES + ['bgp']*N_DEVICES
context     = all_series[:, :CTX_LEN]
ground_truth = all_series[:, CTX_LEN:]

print(f'Series: {all_series.shape[0]} | Context: {CTX_LEN} | Horizon: {PRED_LEN}')
print(pd.Series(series_meta).value_counts().to_dict())

Series: 60 | Context: 512 | Horizon: 24
{'cpu': 20, 'packet_drop': 20, 'bgp': 20}


In [ ]:
import properscoring as ps

def mase(forecast, actual, insample):
    mae        = np.mean(np.abs(forecast - actual))
    naive_err  = np.mean(np.abs(np.diff(insample)))
    return float(mae / (naive_err + 1e-8))

def smape(forecast, actual):
    denom = (np.abs(actual) + np.abs(forecast)) / 2 + 1e-8
    return float(np.mean(np.abs(forecast - actual) / denom) * 100)

def crps_score(samples, actual):
    # samples: (n_samples, horizon), actual: (horizon,)
    scores = [ps.crps_ensemble(actual[t], samples[:, t]) for t in range(len(actual))]
    return float(np.mean(scores))

def evaluate(point_forecasts, sample_forecasts, ground_truth, context, series_meta):
    rows = []
    for i, mtype in enumerate(series_meta):
        rows.append({
            'metric_type': mtype,
            'MASE':  mase(point_forecasts[i],  ground_truth[i], context[i]),
            'sMAPE': smape(point_forecasts[i], ground_truth[i]),
            'CRPS':  crps_score(sample_forecasts[i], ground_truth[i]),
        })
    df = pd.DataFrame(rows)
    return df.groupby('metric_type')[['MASE','sMAPE','CRPS']].mean().round(4)

print('Eval functions ready.')

Eval functions ready.


In [ ]:
!pip install -q --upgrade --force-reinstall chronos-forecasting
import importlib
import chronos
importlib.reload(chronos)
print(chronos.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6

In [ ]:
import subprocess
result = subprocess.run(['pip', 'show', 'chronos-forecasting'], capture_output=True, text=True)
print(result.stdout)

Name: chronos-forecasting
Version: 2.2.2
Summary: Chronos: Pretrained models for time series forecasting
Home-page: https://github.com/amazon-science/chronos-forecasting
Author: 
Author-email: Abdul Fatir Ansari <ansarnd@amazon.com>, Lorenzo Stella <stellalo@amazon.com>, Caner Turkmen <atturkm@amazon.com>, Oleksandr Shchur <shchuro@amazon.com>, Jaris Küken <kuekenj@amazon.com>, Andreas Auer <auerand@amazon.com>
License: Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other en

In [ ]:
!pip install -q chronos-forecasting timesfm properscoring
print('Done.')

Done.


In [ ]:
from chronos import ChronosPipeline
import chronos
print(chronos.__version__)  # should show 2.2.2

2.2.2


In [ ]:
!pip install -q chronos-forecasting timesfm properscoring
print('Done.')

Done.


In [ ]:
import numpy as np
import pandas as pd
import torch
import properscoring as ps
from chronos import ChronosPipeline
import timesfm
import warnings
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print('All imports OK.')

Device: cuda
All imports OK.


In [ ]:
np.random.seed(42)
N_DEVICES, CTX_LEN, PRED_LEN = 20, 512, 24
TOTAL = CTX_LEN + PRED_LEN

def make_cpu(n, length):
    out = []
    for _ in range(n):
        t = np.arange(length)
        base = np.random.uniform(30, 55)
        daily = 15 * np.sin(2 * np.pi * t / 1440)
        noise = np.random.normal(0, 3, length)
        spikes = np.zeros(length)
        spikes[np.random.choice(length, 5, replace=False)] = np.random.uniform(20, 45, 5)
        out.append(np.clip(base + daily + noise + spikes, 0, 100))
    return np.array(out)

def make_packet_drop(n, length):
    out = []
    for _ in range(n):
        base = np.random.exponential(0.5, length)
        for s in np.random.choice(length - 30, 3, replace=False):
            base[s:s+np.random.randint(5,30)] += np.random.uniform(5, 20)
        out.append(np.clip(base + np.abs(np.random.normal(0, 0.2, length)), 0, 100))
    return np.array(out)

def make_bgp(n, length):
    out = []
    for _ in range(n):
        base = np.ones(length) * np.random.choice([30, 60, 90])
        for e in np.random.choice(length, 2, replace=False):
            dur = min(np.random.randint(5,15), length - e)
            base[e:e+dur] = np.random.choice([30, 60, 90, 120])
        out.append(np.clip(base + np.random.normal(0, 1.5, length), 10, 180))
    return np.array(out)

all_series   = np.vstack([make_cpu(N_DEVICES, TOTAL),
                           make_packet_drop(N_DEVICES, TOTAL),
                           make_bgp(N_DEVICES, TOTAL)])
series_meta  = ['cpu']*N_DEVICES + ['packet_drop']*N_DEVICES + ['bgp']*N_DEVICES
context      = all_series[:, :CTX_LEN]
ground_truth = all_series[:, CTX_LEN:]
print(f'Data ready: {all_series.shape[0]} series, context {CTX_LEN}, horizon {PRED_LEN}')

Data ready: 60 series, context 512, horizon 24


In [ ]:
def mase(forecast, actual, insample):
    mae = np.mean(np.abs(forecast - actual))
    naive_err = np.mean(np.abs(np.diff(insample)))
    return float(mae / (naive_err + 1e-8))

def smape(forecast, actual):
    denom = (np.abs(actual) + np.abs(forecast)) / 2 + 1e-8
    return float(np.mean(np.abs(forecast - actual) / denom) * 100)

def crps_score(samples, actual):
    scores = [ps.crps_ensemble(actual[t], samples[:, t]) for t in range(len(actual))]
    return float(np.mean(scores))

def evaluate(point_forecasts, sample_forecasts, ground_truth, context, series_meta):
    rows = []
    for i, mtype in enumerate(series_meta):
        rows.append({
            'metric_type': mtype,
            'MASE':  mase(point_forecasts[i],  ground_truth[i], context[i]),
            'sMAPE': smape(point_forecasts[i], ground_truth[i]),
            'CRPS':  crps_score(sample_forecasts[i], ground_truth[i]),
        })
    df = pd.DataFrame(rows)
    return df.groupby('metric_type')[['MASE','sMAPE','CRPS']].mean().round(4)

print('Eval functions ready.')

Eval functions ready.


In [ ]:
import chronos
print(chronos.__version__)
print(chronos.__file__)

2.2.2
/usr/local/lib/python3.12/dist-packages/chronos/__init__.py


In [ ]:
chronos_model = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map=DEVICE,
    dtype=torch.bfloat16,
)

In [ ]:
print('Loading Chronos-Bolt-Base...')
chronos_model = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map=DEVICE,
    dtype=torch.bfloat16,
)
print('Running forecasts...')

chronos_point, chronos_samples = [], []
for i, series in enumerate(context):
    inp = torch.tensor(series, dtype=torch.float32).unsqueeze(0)
    forecast = chronos_model.predict(inp, prediction_length=PRED_LEN, num_samples=100)
    samples = forecast[0].numpy().astype(float)
    chronos_point.append(samples.mean(axis=0))
    chronos_samples.append(samples)
    if (i+1) % 10 == 0:
        print(f'  {i+1}/60 done')

chronos_point   = np.array(chronos_point)
chronos_samples = np.array(chronos_samples)
chronos_results = evaluate(chronos_point, chronos_samples, ground_truth, context, series_meta)
print('\nChronos results:')
print(chronos_results)

Loading Chronos-Bolt-Base...
Running forecasts...
  10/60 done
  20/60 done
  30/60 done
  40/60 done
  50/60 done
  60/60 done

Chronos results:
               MASE    sMAPE    CRPS
metric_type                         
bgp          0.9473   3.2608  1.1499
cpu          0.8032   5.7494  2.3571
packet_drop  1.2715  65.5500  0.7229


In [ ]:
import inspect
print(inspect.signature(timesfm.ForecastConfig.__init__))

(self, max_context: int = 0, max_horizon: int = 0, normalize_inputs: bool = False, window_size: int = 0, per_core_batch_size: int = 1, use_continuous_quantile_head: bool = False, force_flip_invariance: bool = True, infer_is_positive: bool = True, fix_quantile_crossing: bool = False, return_backcast: bool = False) -> None


In [ ]:
import inspect
print(inspect.signature(tfm.forecast))

(horizon: int, inputs: list[numpy.ndarray]) -> tuple[numpy.ndarray, numpy.ndarray]


In [ ]:
print('Loading TimesFM 2.5...')
config = timesfm.ForecastConfig(max_horizon=PRED_LEN, per_core_batch_size=1)
tfm = timesfm.TimesFM_2p5_200M_torch(config=config)
tfm.compile(forecast_config=config)
tfm.model = tfm.model.to(DEVICE)
print('TimesFM ready. Running forecasts...')

timesfm_point, timesfm_samples = [], []
for i, series in enumerate(context):
    point, quantiles = tfm.forecast(horizon=PRED_LEN, inputs=[np.array(series, dtype=np.float32)])
    p = np.array(point[0][:PRED_LEN])
    q = np.array(quantiles[0][:PRED_LEN, :]).T
    timesfm_point.append(p)
    timesfm_samples.append(q)
    if (i+1) % 10 == 0:
        print(f'  {i+1}/60 done')

timesfm_point = np.array(timesfm_point)
timesfm_samples = np.array(timesfm_samples)
timesfm_results = evaluate(timesfm_point, timesfm_samples, ground_truth, context, series_meta)
print('\nTimesFM results:')
print(timesfm_results)

Loading TimesFM 2.5...
TimesFM ready. Running forecasts...
  10/60 done
  20/60 done
  30/60 done
  40/60 done
  50/60 done
  60/60 done

TimesFM results:
               MASE     sMAPE    CRPS
metric_type                          
bgp          1.1859    4.7768  1.7174
cpu          0.8051    5.7548  2.6625
packet_drop  1.9748  108.2763  1.0995


In [ ]:
summary = pd.DataFrame({
    'Model': ['Chronos-Bolt-Base', 'TimesFM-2.5'],
    'BGP MASE':         [chronos_results.loc['bgp','MASE'],         timesfm_results.loc['bgp','MASE']],
    'CPU MASE':         [chronos_results.loc['cpu','MASE'],         timesfm_results.loc['cpu','MASE']],
    'Pkt Drop MASE':    [chronos_results.loc['packet_drop','MASE'], timesfm_results.loc['packet_drop','MASE']],
    'BGP sMAPE':        [chronos_results.loc['bgp','sMAPE'],        timesfm_results.loc['bgp','sMAPE']],
    'CPU sMAPE':        [chronos_results.loc['cpu','sMAPE'],        timesfm_results.loc['cpu','sMAPE']],
    'Pkt Drop sMAPE':   [chronos_results.loc['packet_drop','sMAPE'],timesfm_results.loc['packet_drop','sMAPE']],
    'BGP CRPS':         [chronos_results.loc['bgp','CRPS'],         timesfm_results.loc['bgp','CRPS']],
    'CPU CRPS':         [chronos_results.loc['cpu','CRPS'],         timesfm_results.loc['cpu','CRPS']],
    'Pkt Drop CRPS':    [chronos_results.loc['packet_drop','CRPS'], timesfm_results.loc['packet_drop','CRPS']],
}).set_index('Model')

print('='*60)
print('BENCHMARK RESULTS — Vigil Telemetry Forecasting')
print('='*60)

print('\n--- MASE (lower is better) ---')
print(summary[['BGP MASE','CPU MASE','Pkt Drop MASE']])

print('\n--- sMAPE (lower is better) ---')
print(summary[['BGP sMAPE','CPU sMAPE','Pkt Drop sMAPE']])

print('\n--- CRPS (lower is better) ---')
print(summary[['BGP CRPS','CPU CRPS','Pkt Drop CRPS']])

# Overall winner per metric
overall = pd.DataFrame({
    'Chronos': [chronos_results['MASE'].mean(), chronos_results['sMAPE'].mean(), chronos_results['CRPS'].mean()],
    'TimesFM': [timesfm_results['MASE'].mean(), timesfm_results['sMAPE'].mean(), timesfm_results['CRPS'].mean()],
}, index=['MASE','sMAPE','CRPS']).round(4)

print('\n--- Overall average across all series ---')
print(overall)
winner = overall.idxmin(axis=1)
print('\nWinner per metric:')
print(winner)

BENCHMARK RESULTS — Vigil Telemetry Forecasting

--- MASE (lower is better) ---
                   BGP MASE  CPU MASE  Pkt Drop MASE
Model                                               
Chronos-Bolt-Base    0.9473    0.8032         1.2715
TimesFM-2.5          1.1859    0.8051         1.9748

--- sMAPE (lower is better) ---
                   BGP sMAPE  CPU sMAPE  Pkt Drop sMAPE
Model                                                  
Chronos-Bolt-Base     3.2608     5.7494         65.5500
TimesFM-2.5           4.7768     5.7548        108.2763

--- CRPS (lower is better) ---
                   BGP CRPS  CPU CRPS  Pkt Drop CRPS
Model                                               
Chronos-Bolt-Base    1.1499    2.3571         0.7229
TimesFM-2.5          1.7174    2.6625         1.0995

--- Overall average across all series ---
       Chronos  TimesFM
MASE    1.0073   1.3219
sMAPE  24.8534  39.6026
CRPS    1.4100   1.8265

Winner per metric:
MASE     Chronos
sMAPE    Chronos
CRPS     Chron